<a href="https://colab.research.google.com/github/Terheer/Cnn/blob/main/basic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import torchvision
import torchvision.transforms as transforms


from torch.utils.data import DataLoader, random_split, Subset

In [ ]:
import zipfile

with zipfile.ZipFile('archive (16).zip', 'r') as zip_ref:
    zip_ref.extractall('extracted_folder')

print("Extraction complete!")

BadZipFile: File is not a zip file

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [ ]:
from torchvision import datasets

root_dir = "/content/extracted_folder/Dataset"


full_aug_dataset = datasets.ImageFolder(root=root_dir, transform=train_transform)
full_eval_dataset = datasets.ImageFolder(root=root_dir, transform=eval_transform)

generator = torch.Generator().manual_seed(42)
train_idx, val_idx, test_idx = random_split(
    range(len(full_aug_dataset)),
    [0.7, 0.15, 0.15],
    generator=generator
)

In [ ]:
train_dataset = Subset(full_aug_dataset, train_idx.indices)
val_dataset = Subset(full_eval_dataset, val_idx.indices)
test_dataset = Subset(full_eval_dataset, test_idx.indices)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

print(f"Classes found: {full_aug_dataset.classes}")
print(f"Train size: {len(train_dataset)} | Val size: {len(val_dataset)} | Test size: {len(test_dataset)}")

Classes found: ['Bus', 'Car', 'Truck', 'motorcycle']
Train size: 280 | Val size: 60 | Test size: 60


In [ ]:
import os
from PIL import Image

#find the first image file in the dataset directory
root_dir = "/content/extracted_folder/Dataset"
found_path = None

for root, dirs, files in os.walk(root_dir):
    for file in files:
        if file.lower().endswith(('.png', '.jpg', '.jpeg')):
            found_path = os.path.join(root, file)
            break
    if found_path:
        break

if found_path:
    print(f"Loading image from: {found_path}")
    img = Image.open(found_path)
    if img.mode == "P" and "transparency" in img.info:
        img = img.convert("RGBA")
else:
    print("No image files found in the dataset directory.")



Loading image from: /content/extracted_folder/Dataset/motorcycle/Image_61.jpg


In [ ]:
import warnings
warnings.filterwarnings(
    "ignore",
    message="Palette images with Transparency expressed in bytes",
    category=UserWarning,
    module="PIL.Image",
)

In [ ]:
class VehicleCNN(nn.Module):

def get_resnet_model(num_classes, fine_tune=True):
    # Load pre-trained ResNet-50
    model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

    # Optional: Freeze early layers if your dataset is small
    if not fine_tune:
        for param in model.parameters():
            param.requires_grad = False

    # Replace the final fully connected layer for your number of classes
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(0.3), # Optional: helps prevent overfitting
        nn.Linear(in_features, num_classes)
    )

    return model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = VehicleCNN(num_classes=4).to(device)
print(model)

VehicleCNN(
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv4): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=50176, out_features=512, bias=True)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc2): Linear(in_features=512, out_features=4, bias=True)
)


In [ ]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
num_epochs = 30
patience = 5               # Number of epochs to wait for improvement before stopping
best_test_acc = 0.0        # Track the highest test accuracy seen
patience_counter = 0       # Counter for tracking epochs without improvement
best_model_path = "best_model.pth"

for epoch in range(num_epochs):
    # --- Training Phase ---
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        total_train += labels.size(0)
        correct_train += predicted.eq(labels).sum().item()

    epoch_loss = running_loss / total_train
    train_acc = 100.0 * correct_train / total_train

    # --- Evaluation Phase ---
    model.eval()
    correct_test = 0
    total_test = 0

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = outputs.max(1)
            total_test += labels.size(0)
            correct_test += predicted.eq(labels).sum().item()

    test_acc = 100.0 * correct_test / total_test

    print(f"Epoch [{epoch+1}/{num_epochs}] | Loss: {epoch_loss:.4f} | Train Acc: {train_acc:.2f}% | Test Acc: {test_acc:.2f}%")

    # --- Checkpointing & Early Stopping Logic ---
    if test_acc > best_test_acc:
        best_test_acc = test_acc
        patience_counter = 0  # Reset counter on improvement
        torch.save(model.state_dict(), best_model_path)
        print(f"  -> Saved new best model weights (Test Acc: {best_test_acc:.2f}%)")
    else:
        patience_counter += 1
        print(f"  -> Early stopping patience: {patience_counter}/{patience}")

    if patience_counter >= patience:
        print(f"\nEarly stopping triggered at epoch {epoch+1}! Best Test Accuracy: {best_test_acc:.2f}%")
        break

# Load the best model weights back into the model for inference/testing
model.load_state_dict(torch.load(best_model_path))

Epoch [1/30] | Loss: 1.6174 | Train Acc: 26.79% | Test Acc: 31.67%
  -> Saved new best model weights (Test Acc: 31.67%)
Epoch [2/30] | Loss: 1.3887 | Train Acc: 28.21% | Test Acc: 26.67%
  -> Early stopping patience: 1/5
Epoch [3/30] | Loss: 1.3443 | Train Acc: 40.00% | Test Acc: 33.33%
  -> Saved new best model weights (Test Acc: 33.33%)
Epoch [4/30] | Loss: 1.2641 | Train Acc: 45.71% | Test Acc: 38.33%
  -> Saved new best model weights (Test Acc: 38.33%)
Epoch [5/30] | Loss: 1.1094 | Train Acc: 53.57% | Test Acc: 38.33%
  -> Early stopping patience: 1/5
Epoch [6/30] | Loss: 1.0086 | Train Acc: 58.57% | Test Acc: 48.33%
  -> Saved new best model weights (Test Acc: 48.33%)
Epoch [7/30] | Loss: 0.9099 | Train Acc: 61.43% | Test Acc: 51.67%
  -> Saved new best model weights (Test Acc: 51.67%)
Epoch [8/30] | Loss: 0.8001 | Train Acc: 71.79% | Test Acc: 51.67%
  -> Early stopping patience: 1/5
Epoch [9/30] | Loss: 0.7770 | Train Acc: 69.64% | Test Acc: 53.33%
  -> Saved new best model weig

<All keys matched successfully>

In [ ]:
torch.save(model.state_dict(), 'train_net.pth')

In [ ]:
from os import pread
correct = 0
total = 0

model.eval()

with torch.no_grad():
  for data in test_loader:
    images, labels = data
    outputs = model(images)
    _, predicted = torch.max(outputs, 1)
    total += labels.size(0)
    correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total

print(f'Accuracy of the network on the 10000 test images: {100 * correct // total} %')

Accuracy of the network on the 10000 test images: 63 %


In [ ]:
import os
from PIL import Image

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


def load_image(image_path):
  img = Image.open(image_path).convert('RGB')
  # Fixed undefined 'new_transform' to 'test_transform'
  img = test_transform(img)
  img = img.unsqueeze(0)
  return img


image_paths = ['piro4d-car-1890494_1920.jpg', 'antriksh-bike-909690_1920.jpg', 'falco-semi-trailer-534577_1920.jpg']
images = []

# Safely load only existing images
for path in image_paths:
  if os.path.exists(path):
    images.append((path, load_image(path).to(device)))
  else:
    print(f"Warning: '{path}' was not found. Please upload it to Colab to make predictions on it.")

# Define class names from dataset
class_names = full_aug_dataset.classes

if images:
  model.eval()
  with torch.no_grad():
    for path, image in images:
      outputs = model(image)
      _, predicted = torch.max(outputs, 1)
      print(f'File: {path} -> Prediction: {class_names[predicted.item()]}')
else:
  print("No valid images to predict. Please upload your images to Colab using the file explorer on the left.")

No valid images to predict. Please upload your images to Colab using the file explorer on the left.
